In [24]:
#imports
import pandas as pd
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
%matplotlib inline

In [25]:
#read and create data frames
df1 = pd.read_csv("dataset/comments1.csv")

In [26]:
#get an idea of how the dataframe looks like
print(df1.head())
print(df1.shape)
print(df1.columns)

              kind  commentId  channelId  videoId  authorId  \
0  youtube#comment    1781382      14492    74288   2032536   
1  youtube#comment     289571      14727    79618   3043229   
2  youtube#comment     569077       3314    51826    917006   
3  youtube#comment    2957962       5008    58298   1853470   
4  youtube#comment     673093      21411     1265   2584166   

                                        textOriginal  parentCommentId  \
0  PLEASE LESBIAN FLAG I BEG YOU \n\nYou would ro...              NaN   
1   Apply mashed potato juice and mixed it with curd        3198066.0   
2                         69 missed calls from mars👽              NaN   
3                                               Baaa              NaN   
4    you look like raven from phenomena raven no cap              NaN   

   likeCount                publishedAt                  updatedAt  
0          0  2023-08-15 21:48:52+00:00  2023-08-15 21:48:52+00:00  
1          0  2023-10-02 13:08:22+00:00  202

In [27]:
#compare textOriginal, likeCount, publishedAt
trend1 = df1[["textOriginal", "likeCount", "publishedAt"]]
trend1.isnull().sum()

textOriginal    46
likeCount        0
publishedAt      0
dtype: int64

In [28]:
#clean up the dataframe
trend1 = trend1.dropna(subset=["textOriginal"])
trend1.head()

,textOriginal,likeCount,publishedAt
0,PLEASE LESBIAN FLAG I BEG YOU \n\nYou would ro...,0,2023-08-15 21:48:52+00:00
1,Apply mashed potato juice and mixed it with curd,0,2023-10-02 13:08:22+00:00
2,69 missed calls from mars👽,0,2024-05-31 12:03:12+00:00
3,Baaa,0,2024-02-13 15:48:37+00:00
4,you look like raven from phenomena raven no cap,0,2020-02-15 22:28:44+00:00


In [29]:
trend1['publishedAt'] = pd.to_datetime(trend1['publishedAt'], format='%Y-%m-%d %H:%M:%S%z', utc=True)
oldest = trend1['publishedAt'].min()
newest = trend1['publishedAt'].max()
oldest 
newest  

Timestamp('2025-07-20 15:09:26+0000', tz='UTC')

In [30]:
first_date = trend1["publishedAt"].min()
trend1["days_since_first"] = (trend1["publishedAt"] - first_date).dt.days.astype(float)

In [31]:
filtered_likes_df = trend1[trend1["likeCount"] > 0]
average_like_count1 = filtered_likes_df["likeCount"].mean()
successful_comments_1 = trend1[trend1["likeCount"] >= average_like_count1]

In [32]:
#input the filtered dataframe that only has higher than average like counts for the parameter
def textOriginalAnalysis(df)-> dict:
    dfT = df["textOriginal"] #dfT short for dfText
    textArray = []
    stop_words = set(stopwords.words('english'))
    for i in dfT:  
        # ?:^ checks for if it is the start, can accept no whitespace if at the start
        words = re.findall(r'(?:^|\s)([1-9][0-9]*|[a-zA-Z]+)(?=\s|$)', i.lower(), flags=re.IGNORECASE)

        filtered_words = [word for word in words if word not in stop_words]

        textArray.extend(filtered_words)

    frequency_dict = Counter(textArray)
    topTenPercent = int(len(frequency_dict) / 10)   
    maxcount = frequency_dict.most_common(1)[0][1] #the count of the most common word of all
    
        # Function to convert a single comment into numeric value
    def comment_to_value(comment: str) -> float:
        words = re.findall(r'(?:^|\s)([1-9][0-9]*|[a-zA-Z]+)(?=\s|$)', 
                           str(comment).lower(), flags=re.IGNORECASE)
        filtered_words = [word for word in words if word not in stop_words]
        if not filtered_words:
            return 0.0
        # Average normalized frequency
        values = [frequency_dict[w] / maxcount for w in filtered_words if w in frequency_dict]
        return sum(values) / len(values) if values else 0.0

    # Add new column
    df = df.copy()
    df["commentValue"] = dfT.apply(comment_to_value)

    return df


analyzed_words = textOriginalAnalysis(successful_comments_1)

    

In [34]:
analyzed_words

,textOriginal,likeCount,publishedAt,days_since_first,commentValue
136,How do you achieve that slick back? 🧐,684,2025-01-28 09:03:05+00:00,1850.0,0.009693
452,"“If it rains, I’m ruined” was so REALL😭😮😢",88,2024-07-13 05:19:05+00:00,1651.0,0.000000
533,Can we please go back to Classic beauty?,75,2024-06-11 04:17:52+00:00,1619.0,0.084006
573,POPULAR..YOURE GONNA BE POPULARR!,701,2024-11-25 15:03:24+00:00,1787.0,0.070275
744,"Bro my skincare routine is soap, water and a t...",808,2023-04-29 16:40:35+00:00,1211.0,0.039849
...,...,...,...,...,...
999631,gorgeous darling <3,245,2021-09-24 20:39:08+00:00,629.0,0.088853
999799,Me who goes anywhere with just a face wash,304,2023-05-02 13:59:24+00:00,1213.0,0.088651
999806,"I feel like U have more of a wave pattern, but...",2262,2024-11-22 03:04:23+00:00,1783.0,0.222132
999821,Name the girls id you will get all information...,57,2023-08-03 20:24:14+00:00,1307.0,0.072429
